# Hypothesis Testing for Insurance Risk Analytics

## Objective
This notebook statistically tests key business hypotheses about insurance risk drivers to support segmentation and pricing decisions.

### Risk Metrics
- **Claim Frequency** → proportion of policies with at least one claim
- **Claim Severity** → average claim amount
- **Margin** → TotalPremium − TotalClaims

### Hypotheses Tested
1. H₀: There are no risk differences across provinces.
2. H₀: There are no risk differences between zip codes.
3. H₀: There is no significant margin difference between zip codes.
4. H₀: There is no significant risk difference between Women and Men.

In [1]:
import pandas as pd
import numpy as np

import warnings
warnings.filterwarnings("ignore")

### Load Data

In [2]:
try:
    df = pd.read_csv("../data/MachineLearningRating_V3/cleaned_insurance_data.csv")
    print("Data loaded successfully")
    print("Shape:", df.shape)

except FileNotFoundError:
    print("[ERROR] File not found. Check file path.")

except Exception as e:
    print(f"[ERROR] Failed to load data: {e}")

Data loaded successfully
Shape: (1000098, 54)


### Preview Data

In [3]:
try:
    display(df.head())
    print(df.info())

except Exception as e:
    print(f"[ERROR] Failed to preview data: {e}")

,UnderwrittenCoverID,PolicyID,TransactionMonth,IsVATRegistered,Citizenship,LegalType,Title,Language,Bank,AccountType,...,CoverType,CoverGroup,Section,Product,StatutoryClass,StatutoryRiskType,TotalPremium,TotalClaims,LossRatio,Margin
0,145249,12827,2015-03-01,True,,Close Corporation,Mr,English,First National Bank,Current account,...,Windscreen,Comprehensive - Taxi,Motor Comprehensive,Mobility Metered Taxis: Monthly,Commercial,IFRS Constant,21.929825,0.0,0.0,21.929825
1,145249,12827,2015-05-01,True,,Close Corporation,Mr,English,First National Bank,Current account,...,Windscreen,Comprehensive - Taxi,Motor Comprehensive,Mobility Metered Taxis: Monthly,Commercial,IFRS Constant,21.929825,0.0,0.0,21.929825
2,145249,12827,2015-07-01,True,,Close Corporation,Mr,English,First National Bank,Current account,...,Windscreen,Comprehensive - Taxi,Motor Comprehensive,Mobility Metered Taxis: Monthly,Commercial,IFRS Constant,0.000000,0.0,NaN,0.000000
3,145255,12827,2015-05-01,True,,Close Corporation,Mr,English,First National Bank,Current account,...,Own Damage,Comprehensive - Taxi,Motor Comprehensive,Mobility Metered Taxis: Monthly,Commercial,IFRS Constant,512.848070,0.0,0.0,512.848070
4,145255,12827,2015-07-01,True,,Close Corporation,Mr,English,First National Bank,Current account,...,Own Damage,Comprehensive - Taxi,Motor Comprehensive,Mobility Metered Taxis: Monthly,Commercial,IFRS Constant,0.000000,0.0,NaN,0.000000


<class 'pandas.DataFrame'>
RangeIndex: 1000098 entries, 0 to 1000097
Data columns (total 54 columns):
 #   Column                    Non-Null Count    Dtype  
---  ------                    --------------    -----  
 0   UnderwrittenCoverID       1000098 non-null  int64  
 1   PolicyID                  1000098 non-null  int64  
 2   TransactionMonth          1000098 non-null  str    
 3   IsVATRegistered           1000098 non-null  bool   
 4   Citizenship               1000098 non-null  str    
 5   LegalType                 1000098 non-null  str    
 6   Title                     1000098 non-null  str    
 7   Language                  1000098 non-null  str    
 8   Bank                      854137 non-null   str    
 9   AccountType               959866 non-null   str    
 10  MaritalStatus             991839 non-null   str    
 11  Gender                    990562 non-null   str    
 12  Country                   1000098 non-null  str    
 13  Province                  1000098 non-

## Feature Engineering

Create derived variables needed for hypothesis testing:
- **HasClaim** for claim frequency
- **ClaimSeverity** for claim severity
- **Margin** for profitability

In [4]:
try:
    df = df.copy()

    df["HasClaim"] = (df["TotalClaims"] > 0).astype(int)

    df["ClaimSeverity"] = df["TotalClaims"]

    df["Margin"] = df["TotalPremium"] - df["TotalClaims"]

    print("Derived columns created successfully")

except Exception as e:
    print(f"[ERROR] Feature engineering failed: {e}")

Derived columns created successfully


## Import Hypothesis Testing Functions (from src)

In [6]:
try:
    import sys
    sys.path.append("..")
    from src.hypothesis_tests import (
        run_ttest,
        run_chi2_test,
        hypothesis_decision
    )

    print("Functions imported successfully")

except Exception as e:
    print(f"[ERROR] Import failed: {e}")

Functions imported successfully


## Hypothesis 1: Risk Differences Across Provinces

**KPI:** Claim Severity  
**Test:** Independent t-test

### Province Test

In [7]:
try:
    province_a = df[df["Province"] == "Gauteng"]["ClaimSeverity"]
    province_b = df[df["Province"] == "Western Cape"]["ClaimSeverity"]

    p1 = run_ttest(province_a, province_b)
    d1 = hypothesis_decision(p1)

    print("P-value:", p1)
    print("Decision:", d1)

except Exception as e:
    print(f"[ERROR] Province hypothesis test failed: {e}")

P-value: 0.06215231452280036
Decision: Fail to Reject H0


## Hypothesis 2: Risk Differences Between Zip Codes

**KPI:** Claim Severity  
**Test:** Independent t-test

### Zip Risk Test

In [8]:
try:
    zip_a = df[df["PostalCode"] == 6200]["ClaimSeverity"]
    zip_b = df[df["PostalCode"] == 6001]["ClaimSeverity"]

    p2 = run_ttest(zip_a, zip_b)
    d2 = hypothesis_decision(p2)

    print("P-value:", p2)
    print("Decision:", d2)

except Exception as e:
    print(f"[ERROR] Zip code risk test failed: {e}")

P-value: 0.3175476759102478
Decision: Fail to Reject H0


## Hypothesis 3: Margin Differences Between Zip Codes

**KPI:** Margin  
**Test:** Independent t-test

### Zip Margin Test

In [9]:
try:
    zip_margin_a = df[df["PostalCode"] == 6200]["Margin"]
    zip_margin_b = df[df["PostalCode"] == 6001]["Margin"]

    p3 = run_ttest(zip_margin_a, zip_margin_b)
    d3 = hypothesis_decision(p3)

    print("P-value:", p3)
    print("Decision:", d3)

except Exception as e:
    print(f"[ERROR] Zip code margin test failed: {e}")

P-value: 0.4777766509585694
Decision: Fail to Reject H0


## Hypothesis 4: Risk Difference Between Women and Men

**KPI:** Claim Frequency  
**Test:** Chi-square

### Gender Test

In [10]:
try:
    gender_df = df[df["Gender"].isin(["Male", "Female"])]

    p4 = run_chi2_test(
        gender_df,
        "Gender",
        "HasClaim"
    )

    d4 = hypothesis_decision(p4)

    print("P-value:", p4)
    print("Decision:", d4)

except Exception as e:
    print(f"[ERROR] Gender hypothesis test failed: {e}")

P-value: 0.9514644755420456
Decision: Fail to Reject H0


## Results Summary Table

In [11]:
try:
    results = pd.DataFrame({
        "Hypothesis": [
            "Province Risk Difference",
            "Zip Code Risk Difference",
            "Zip Code Margin Difference",
            "Gender Risk Difference"
        ],
        "Test Used": [
            "t-test",
            "t-test",
            "t-test",
            "Chi-square"
        ],
        "P-value": [
            p1,
            p2,
            p3,
            p4
        ],
        "Decision": [
            d1,
            d2,
            d3,
            d4
        ]
    })

    display(results)

except Exception as e:
    print(f"[ERROR] Failed to create results table: {e}")

,Hypothesis,Test Used,P-value,Decision
0,Province Risk Difference,t-test,0.062152,Fail to Reject H0
1,Zip Code Risk Difference,t-test,0.317548,Fail to Reject H0
2,Zip Code Margin Difference,t-test,0.477777,Fail to Reject H0
3,Gender Risk Difference,Chi-square,0.951464,Fail to Reject H0


# Business Recommendations

Based on the hypothesis testing results, no statistically significant differences were found across provinces, zip codes, or gender. This leads to the following business recommendations for AlphaCare Insurance Solutions (ACIS):

## 1. Avoid Demographic-Based Pricing
- Gender does not significantly impact claim frequency.
- Therefore, gender should not be used as a pricing factor in underwriting or premium calculation.

## 2. Reconsider Geographic Segmentation
- Province and zip code do not show statistically significant differences in risk or profitability.
- Geographic location alone should not be used as a primary driver for differentiated pricing strategies.

## 3. Focus on Stronger Risk Predictors
- Since demographic and geographic variables are weak predictors, ACIS should prioritize:
  - Vehicle make and model
  - Vehicle type and characteristics
  - Policy structure and coverage type
  - Historical claim behavior

## 4. Move Toward Data-Driven Pricing Models
- Traditional segmentation based on simple grouping variables is insufficient.
- ACIS should invest in predictive modeling approaches (e.g., regression, random forests, gradient boosting) to capture complex risk patterns.

## 5. Improve Feature Engineering Strategy
- Future modeling should focus on engineered risk indicators such as:
  - Claim frequency per policy
  - Claim severity trends
  - Exposure-based normalization metrics

## 6. Regulatory and Fairness Considerations
- Since gender and geography are not statistically significant, excluding them from pricing models can also support fairness and regulatory compliance.

## Final Insight
Overall, the analysis suggests that insurance risk in this dataset is not strongly driven by demographic or geographic segmentation. More granular and behavioral features should be prioritized to improve pricing accuracy and portfolio profitability.